# EcoMarket - Atención al Cliente con RAG (Ollama + LangChain)

Este notebook implementa un sistema de **Retrieval-Augmented Generation (RAG)** para el caso **EcoMarket**, un asistente virtual de atención al cliente. El sistema utiliza:

- **[Ollama](https://ollama.com)** como servidor local del LLM (`llama3.2:3b`).
- **[LangChain](https://www.langchain.com)** como framework para orquestar el pipeline RAG.
- **FAISS** como vectorstore para la indexación y recuperación de documentos.
- **HuggingFace embeddings** multilingües para representar los textos.

### Fuentes de conocimiento
El asistente de EcoMarket responde consultas a partir de dos fuentes locales:

1. `data/pedidos.json` — Estado y detalle de los pedidos de los clientes.
2. `data/politica_devoluciones.txt` — Política oficial de devoluciones de EcoMarket.

Los prompts usados por el modelo se cargan desde la carpeta `prompts/`:
- `prompts/prompt_pedido.txt`
- `prompts/prompt_devolucion.txt`

---

### Requisitos previos (ejecución local en Windows)

1. **Python 3.10+** con un entorno virtual activo como kernel del notebook.
2. **Ollama** instalado y corriendo en `http://localhost:11434` (ver sección 2).
3. Ejecutar el notebook **desde la raíz del proyecto** (donde viven `data/` y `prompts/`) para que las rutas relativas resuelvan correctamente.
4. Conexión a internet en la primera ejecución: se descarga el modelo `llama3.2:3b` (~2 GB) y los embeddings `intfloat/multilingual-e5-large` (~1.3 GB). En corridas posteriores todo se lee desde caché.

## 0. Instalación de dependencias Python

Ejecuta esta celda una sola vez por entorno. Usa `%pip` para garantizar que las librerías se instalen en el mismo Python que usa el kernel del notebook.

Alternativa equivalente desde terminal: `pip install -r requirements.txt`.

In [1]:
%pip install -q \
    "langchain>=0.3,<0.4" \
    "langchain-community>=0.3,<0.4" \
    "langchain-ollama>=0.2,<0.3" \
    "langchain-huggingface>=0.1,<0.2" \
    ollama \
    faiss-cpu \
    sentence-transformers \
    gradio

import warnings
warnings.filterwarnings('ignore')

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\josue\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 1. Carga de documentos locales

Cargamos las dos fuentes locales y las convertimos en `Document` de LangChain. Cada documento lleva metadatos (`source`, `tipo`) para poder citarlo en las respuestas.

In [2]:
import json
from pathlib import Path
from langchain.schema import Document

DATA_DIR = Path('data')
PEDIDOS_PATH = DATA_DIR / 'pedidos.json'
POLITICA_PATH = DATA_DIR / 'politica_devoluciones.txt'

def cargar_pedidos(path: Path):
    with open(path, 'r', encoding='utf-8') as f:
        pedidos = json.load(f)
    docs = []
    for p in pedidos:
        contenido = (
            f"Pedido {p['tracking_number']}\n"
            f"Estado: {p['estado']}\n"
            f"Fecha estimada de entrega: {p['fecha_entrega']}\n"
            f"Detalle: {p['detalle']}"
        )
        docs.append(Document(
            page_content=contenido,
            metadata={
                'source': str(path),
                'tipo': 'pedido',
                'tracking_number': p['tracking_number'],
            },
        ))
    return docs

def cargar_politica(path: Path):
    texto = path.read_text(encoding='utf-8')
    return [Document(
        page_content=texto,
        metadata={'source': str(path), 'tipo': 'politica_devoluciones'},
    )]

docs_pedidos = cargar_pedidos(PEDIDOS_PATH)
docs_politica = cargar_politica(POLITICA_PATH)
documents = docs_pedidos + docs_politica

print(f"Pedidos cargados: {len(docs_pedidos)}")
print(f"Documentos de política: {len(docs_politica)}")
print(f"Total de documentos: {len(documents)}")

Pedidos cargados: 10
Documentos de política: 1
Total de documentos: 11


## 2. Instalación y arranque de Ollama (Windows)

Ollama se instala **fuera del notebook**, una sola vez:

1. Descarga el instalador desde https://ollama.com/download/windows y ejecútalo.
2. Tras instalar, Ollama queda corriendo como servicio en `http://localhost:11434`. Si no lo está, abre una terminal y ejecuta `ollama serve`.
3. Verifica en una terminal nueva: `ollama --version`.

> En Linux/macOS puedes usar `curl -fsSL https://ollama.com/install.sh | sh`. En Windows el instalador oficial es la forma soportada.

La siguiente celda descarga el modelo `llama3.2:3b` (~2 GB la primera vez). Si ya lo descargaste antes, Ollama reportará que el modelo ya existe.

Descargamos el modelo `llama3.2:3b` para usarlo como LLM del asistente de EcoMarket.

In [3]:
!ollama pull llama3.2:3b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 288 KB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 1.3 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 1.8 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 2.7 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 3.8 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 4.3 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 5.6 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕         

## 3. LLM con LangChain + Ollama
Conectamos LangChain al modelo servido por Ollama.

In [4]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.2:3b", validate_model_on_init=True)

llm.invoke("Responde brevemente: ¿qué es EcoMarket?").content

'EcoMarket es una plataforma de intercambio de bienes y servicios sostenibles que conecta a proveedores y consumidores que comparten valores ecológicos y sociales. Su objetivo es promover la economía circular y el consumo responsable, alentando a los usuarios a optar por productos y servicios que sean más sostenibles con el medio ambiente.'

## 4. Construcción del vectorstore (FAISS)

- Dividimos los documentos en chunks para mejorar el retrieval.
- Generamos embeddings con un modelo multilingüe.
- Indexamos todo en FAISS.

> **Primera ejecución:** se descarga `intfloat/multilingual-e5-large` (~1.3 GB) desde HuggingFace y se cachea en `~/.cache/huggingface`. Puede tardar varios minutos. En corridas siguientes el índice se carga directamente desde `./faiss_index_ecomarket/`.

In [5]:
import os
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter

embeddings = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-large")

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)
print(f"Chunks generados: {len(chunks)}")

index_path = './faiss_index_ecomarket'
if os.path.exists(index_path):
    vectorstore = FAISS.load_local(index_path, embeddings, allow_dangerous_deserialization=True)
else:
    vectorstore = FAISS.from_documents(chunks, embeddings)
    vectorstore.save_local(index_path)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3256.52it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Chunks generados: 11


## 5. Prompts del asistente

Los prompts se cargan directamente desde los archivos en la carpeta `prompts/` para mantener una única fuente de verdad.

In [6]:
from pathlib import Path

PROMPTS_DIR = Path('prompts')

prompt_pedido_tpl = (PROMPTS_DIR / 'prompt_pedido.txt').read_text(encoding='utf-8')
prompt_devolucion_tpl = (PROMPTS_DIR / 'prompt_devolucion.txt').read_text(encoding='utf-8')

print('--- prompt_pedido.txt ---')
print(prompt_pedido_tpl)
print('--- prompt_devolucion.txt ---')
print(prompt_devolucion_tpl)

--- prompt_pedido.txt ---
Actúa como un agente experto de servicio al cliente de EcoMarket, especializado en pedidos y logística.

### Contexto del sistema:
Tienes acceso a la siguiente información de pedidos:
{data}

### Tarea:
El cliente quiere conocer el estado de su pedido con número: {tracking_number}

### Instrucciones:
- Responde únicamente con base en la información proporcionada.
- NO inventes datos ni hagas suposiciones.
- Si el pedido no existe, indícalo claramente.
- Usa un tono amable, profesional y cercano.
- Incluye:
  1. Estado del pedido
  2. Fecha estimada de entrega
  3. Explicación breve del estado
- Si el pedido está retrasado:
  - Ofrece disculpas
  - Explica brevemente la causa
- Mantén la respuesta clara y breve (máximo 5 líneas).

### Formato de respuesta esperado:
Saludo breve
Estado del pedido
Fecha de entrega
Mensaje adicional (si aplica)

### Respuesta:

--- prompt_devolucion.txt ---
Actúa como un agente de servicio al cliente de EcoMarket experto en políti

## 6. Pipeline RAG para EcoMarket

Implementamos un router sencillo que:

1. Decide si la consulta es sobre un **pedido** o sobre **devoluciones**.
2. Recupera el contexto relevante del vectorstore.
3. Aplica el prompt correspondiente (`prompt_pedido.txt` o `prompt_devolucion.txt`).
4. Invoca al LLM y devuelve la respuesta junto con las fuentes.

In [7]:
import re
from langchain.prompts import PromptTemplate

prompt_pedido = PromptTemplate.from_template(prompt_pedido_tpl)
prompt_devolucion = PromptTemplate.from_template(prompt_devolucion_tpl)

def extraer_tracking(pregunta: str) -> str | None:
    m = re.search(r'\b(\d{3,})\b', pregunta)
    return m.group(1) if m else None

def es_consulta_pedido(pregunta: str) -> bool:
    claves = ['pedido', 'orden', 'tracking', 'envío', 'envio', 'estado']
    return any(k in pregunta.lower() for k in claves) or extraer_tracking(pregunta) is not None

def formatear_contexto(docs):
    return "\n\n".join(d.page_content for d in docs)

def responder(pregunta: str):
    if es_consulta_pedido(pregunta):
        tracking = extraer_tracking(pregunta) or 'N/A'
        docs = retriever.invoke(pregunta)
        docs_pedido = [d for d in docs if d.metadata.get('tipo') == 'pedido'] or docs
        contexto = formatear_contexto(docs_pedido)
        prompt_final = prompt_pedido.format(data=contexto, tracking_number=tracking)
        fuentes = docs_pedido
    else:
        docs = retriever.invoke(pregunta)
        docs_pol = [d for d in docs if d.metadata.get('tipo') == 'politica_devoluciones'] or docs
        contexto = formatear_contexto(docs_pol)
        prompt_final = prompt_devolucion.format(politica=contexto, pregunta=pregunta)
        fuentes = docs_pol

    respuesta = llm.invoke(prompt_final).content
    return {"answer": respuesta, "sources": fuentes}

def imprimir_respuesta(resultado):
    print(resultado['answer'])
    print("\nFuentes:")
    for i, d in enumerate(resultado['sources'], 1):
        src = d.metadata.get('source', 'desconocido')
        tipo = d.metadata.get('tipo', 'N/A')
        extra = d.metadata.get('tracking_number', '')
        etiqueta = f"{tipo}" + (f" #{extra}" if extra else "")
        print(f"- [{i}] {etiqueta} ({src})")

## 7. Preguntas de prueba

In [8]:
resultado_1 = responder("¿Cuál es el estado de mi pedido 1003?")
imprimir_respuesta(resultado_1)

Hola, le agradezco que nos contactó sobre su pedido con número 1003. 

Estado: Retrasado
Fecha estimada de entrega: 2026-04-18
Tenemos una alta demanda logística en este momento y eso ha causado un retraso en la entrega de su pedido.

Si necesita más información o ayuda, por favor no dude en hacérmelo saber.

Fuentes:
- [1] pedido #1001 (data\pedidos.json)
- [2] pedido #1002 (data\pedidos.json)
- [3] pedido #1003 (data\pedidos.json)
- [4] pedido #1005 (data\pedidos.json)


In [9]:
resultado_2 = responder("¿Puedo devolver un producto de higiene personal?")
imprimir_respuesta(resultado_2)

No aplica. 

La política establece que no se aceptan devoluciones de productos perecederos, y como la higiene personal es un producto categorizado como tal, no cumple con los requisitos para una devolución.

Puede contactar a nuestro equipo de atención al cliente para obtener más información sobre el proceso de devolución.

Fuentes:
- [1] politica_devoluciones (data\politica_devoluciones.txt)


## Conclusiones

- LangChain + Ollama permiten construir un asistente RAG local sin depender de APIs externas.
- FAISS indexa los documentos de EcoMarket (`pedidos.json` y `politica_devoluciones.txt`) y responde con contexto verificable.
- Los prompts se mantienen desacoplados en `prompts/`, lo que facilita iterar sobre el tono y las reglas del asistente sin modificar el código.
- El router simple (pedidos vs. devoluciones) mejora la precisión al aplicar el prompt correcto según la intención del cliente.